In [1]:
import tifffile
import torch
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.ndimage as ndi
from collections import defaultdict
from cellpose.models import CellposeModel
from cellpose.utils import remove_edge_masks
import cellutils as cu

%load_ext autoreload
%autoreload 2

In [2]:
modelpath = Path(
    "/home/starrluxton/BurgessLab/yeast_monolayer_project/models"
)
modelname = "cp4_20260724_t02"

In [13]:
# pair up brightfield and DAPI images
imgpath = Path("/home/starrluxton/BurgessLab/yeast_monolayer_project/yeast_images/wild-type")
brightfield_path = imgpath / "focal_slices_cellpose_training"
deconvolved_path = imgpath / "projected_DAPI"

brightfield_flist = list(brightfield_path.glob("[!.]*_wt_*.tif*"))
deconvolved_flist = list(deconvolved_path.glob("[!.]*decon.tif*"))

brightfield_map = {f.stem.split("_")[-1]: f.name for f in brightfield_flist}
decon_map = {f.stem.split("_")[-2]: f.name for f in deconvolved_flist}

matched_flist = {
    key: (brightfield_map[key], decon_map[key])
    for key in brightfield_map
    if key in decon_map
}

print(brightfield_map)

{'F1': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1.tiff', 'F4': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4.tiff', 'F0': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0.tiff', 'F6': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6.tiff', 'F2': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2.tiff', 'F3': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3.tiff', 'F5': '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F5.tiff'}


In [14]:
for key, val in matched_flist.items():
    print(key, val)

F1 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1_decon.tif')
F4 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4_decon.tif')
F0 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0_decon.tif')
F6 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6_decon.tif')
F2 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2_decon.tif')
F3 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3.tiff', '2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3_decon.tif')
F5 ('2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F5.tiff', '2026-

In [15]:
# load cellpose model
cpmodel = CellposeModel(
    pretrained_model=modelpath / modelname, device=torch.device("cuda")
)

In [16]:
output_path = Path("/home/starrluxton/BurgessLab/yeast_monolayer_project/yeast_images/crops")
sample_name = "wildtype_01"

for key, (bf_fname, dapi_fname) in matched_flist.items():
    print(f"Working on key {key}...")
    print(bf_fname, dapi_fname)
    bfimg = tifffile.imread(brightfield_path / bf_fname)
    dapiimg = tifffile.imread(deconvolved_path / dapi_fname)
    masks, flows, styles = cpmodel.eval([bfimg], diameter=50.0)
    masks = remove_edge_masks(masks[0])
    crop_path = Path(f"{sample_name}_{key}")
    df = cu.export_fov(masks, bfimg, dapiimg, sample_name, output_path / crop_path)

Working on key F1...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F1_decon.tif
Working on key F4...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F4_decon.tif
Working on key F0...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F0_decon.tif
Working on key F6...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F6_decon.tif
Working on key F2...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F2_decon.tif
Working on key F3...
2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3.tiff 2026-07-20_260717_wt_ref_White Light - Brightfield_EPI - 405_F3_decon.tif
Working on key F